In [1]:
import os, json, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             confusion_matrix, roc_curve)
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Paths ─────────────────────────────────────────────────
BASE = "/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset"
CSV  = f"{BASE}/Camelyon16.csv"
OUT  = "/kaggle/working"
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT}/plots",       exist_ok=True)
os.makedirs(f"{OUT}/metrics",     exist_ok=True)

# ── Hyperparameters  ───────
MAX_PATCHES = 512
SEED        = 42
EPOCHS      = 20
LR          = 1e-4
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print("Settings: EPOCHS=20, test_size=0.25, lam=0.005 (Run A settings)")

Device : cuda
PyTorch: 2.10.0+cu128
Settings: EPOCHS=20, test_size=0.25, lam=0.005 (Run A settings)


In [2]:
class Camelyon16BagDataset(Dataset):
    def __init__(self, records, base_dir, max_patches=512, seed=42):
        self.records     = records
        self.base_dir    = base_dir
        self.max_patches = max_patches
        self.rng         = np.random.RandomState(seed)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rel_path, label = self.records[idx]
        rel_path  = rel_path.replace("datasets/Camelyon16/", "")
        full_path = os.path.join(self.base_dir, rel_path)
        feats = torch.tensor(
            np.loadtxt(full_path, delimiter=',', skiprows=1),
            dtype=torch.float32)
        N = feats.shape[0]
        if N > self.max_patches:
            chosen = self.rng.choice(N, self.max_patches, replace=False)
            feats  = feats[chosen]
        return {"features": feats,
                "label":    torch.tensor(label, dtype=torch.long)}

def bag_collate(batch):
    return {"features": [b["features"] for b in batch],
            "labels":   torch.stack([b["label"] for b in batch])}

df_meta         = pd.read_csv(CSV)
df_meta.columns = ["path", "label"]
df_meta         = df_meta[df_meta["label"].isin([0,1])].reset_index(drop=True)
records         = list(zip(df_meta["path"], df_meta["label"]))

# RUN A split: 25% test = ~100 slides
train_val, test_rec = train_test_split(
    records, test_size=0.25, random_state=SEED,
    stratify=[r[1] for r in records])
train_rec, val_rec = train_test_split(
    train_val, test_size=0.15, random_state=SEED,
    stratify=[r[1] for r in train_val])

print(f"Train: {len(train_rec)} | Val: {len(val_rec)} | Test: {len(test_rec)}")
print(f"Train pos: {sum(r[1] for r in train_rec)}")
print(f"Val   pos: {sum(r[1] for r in val_rec)}")
print(f"Test  pos: {sum(r[1] for r in test_rec)}")

train_ds = Camelyon16BagDataset(train_rec, BASE, MAX_PATCHES, SEED)
val_ds   = Camelyon16BagDataset(val_rec,   BASE, MAX_PATCHES, SEED)
test_ds  = Camelyon16BagDataset(test_rec,  BASE, MAX_PATCHES, SEED+1)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=bag_collate, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          collate_fn=bag_collate, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          collate_fn=bag_collate, num_workers=2)

print(f"\nSample bag shape: {train_ds[0]['features'].shape}")
print("Dataset OK")

Train: 254 | Val: 45 | Test: 100
Train pos: 102
Val   pos: 18
Test  pos: 40

Sample bag shape: torch.Size([512, 512])
Dataset OK


In [3]:
COLORS = {
    'ABMIL':                  '#2196F3',
    'CLAM-SB':                '#4CAF50',
    'VAEABMIL':               '#FF9800',
    'eval_nogate (No Gating)':'#9C27B0',
    'URAT-MIL (Proposed)':    '#F44336',
}

def fname(name):
    return name.replace(' ','_').replace('(','').replace(')','').replace('/','_')

def plot_training_curves(losses, aucs, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))
    ep = range(1, len(losses)+1)
    ax1.plot(ep, losses, color=COLORS.get(name,'gray'), linewidth=2)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss')
    ax1.set_title(f'Training Loss — {name}'); ax1.grid(alpha=0.3)
    ax2.plot(ep, aucs, color='#F44336', linewidth=2)
    best_ep = int(np.argmax(aucs)) + 1
    ax2.axvline(best_ep, color='gray', linestyle='--', alpha=0.7,
                label=f'Best Ep={best_ep}: {max(aucs):.4f}')
    ax2.legend(fontsize=8)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val AUC')
    ax2.set_title(f'Validation AUC — {name}'); ax2.grid(alpha=0.3)
    plt.tight_layout()
    path = f"{OUT}/plots/training_{fname(name)}.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_confusion(tn, fp, fn, tp, name):
    cm = np.array([[tn,fp],[fn,tp]])
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Predicted\nNormal','Predicted\nTumor'], fontsize=11)
    ax.set_yticklabels(['Actual\nNormal','Actual\nTumor'], fontsize=11)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    fontsize=16, fontweight='bold',
                    color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set_title(f'Confusion Matrix — {name}', fontsize=12)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    path = f"{OUT}/plots/confusion_{fname(name)}.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_all_roc(roc_dict):
    fig, ax = plt.subplots(figsize=(7,6))
    for name,(fpr,tpr,auc_val) in roc_dict.items():
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})",
                color=COLORS.get(name,'gray'), linewidth=2)
    ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves — All Models (CAMELYON16)', fontsize=13)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    path = f"{OUT}/plots/roc_all_models.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_reliability(rel_dict):
    n   = len(rel_dict)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax,(name,(fp2,mc,ece)) in zip(axes, rel_dict.items()):
        ax.plot(mc, fp2, 'o-', color=COLORS.get(name,'gray'), linewidth=2)
        ax.plot([0,1],[0,1],'k--', linewidth=1)
        ax.set_title(f"{name}\nECE={ece:.4f}", fontsize=9)
        ax.set_xlabel('Mean Confidence'); ax.set_ylabel('Fraction Positive')
        ax.grid(alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.suptitle('Reliability Diagrams — All Models', fontsize=12, y=1.02)
    plt.tight_layout()
    path = f"{OUT}/plots/reliability_all.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_ablation_bar(results):
    names = [r['Model'] for r in results]
    aucs  = [r['AUC']   for r in results]
    eces  = [r['ECE']   for r in results]
    accs  = [r['Accuracy'] for r in results]
    x     = np.arange(len(names))
    fig, axes = plt.subplots(1, 3, figsize=(15,5))
    for ax, vals, ylabel, title, ylim in [
        (axes[0], aucs,  'AUC',          'AUC Comparison',      (0.85,1.0)),
        (axes[1], eces,  'ECE  (lower)','ECE Comparison',      (0, 0.55)),
        (axes[2], accs,  'Accuracy (%)', 'Accuracy Comparison', (80, 100)),
    ]:
        bars = ax.bar(x, vals, color=[COLORS.get(n,'gray') for n in names], alpha=0.85)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
        ax.set_ylim(ylim)
        ax.grid(axis='y', alpha=0.3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.002 if ylabel!='Accuracy (%)' else bar.get_height()+0.2,
                    f'{v:.4f}' if ylabel!='Accuracy (%)' else f'{v:.1f}%',
                    ha='center', va='bottom', fontsize=7)
    plt.suptitle('Model Comparison — CAMELYON16 Test Set', fontsize=13)
    plt.tight_layout()
    path = f"{OUT}/plots/ablation_comparison.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_uncertainty(u_alea, u_epis, labels, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))
    labels_arr = np.array(labels)
    for cls, color, label in [(0,'#2196F3','Normal'),(1,'#F44336','Tumor')]:
        mask = labels_arr == cls
        if mask.sum() > 0:
            ax1.hist(np.array(u_alea)[mask], bins=15, alpha=0.6,
                     color=color, label=label, density=True)
            ax2.hist(np.array(u_epis)[mask], bins=15, alpha=0.6,
                     color=color, label=label, density=True)
    ax1.set_title('Aleatoric Uncertainty (U_alea)', fontsize=11)
    ax1.set_xlabel('U_alea'); ax1.set_ylabel('Density')
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.set_title('Epistemic Uncertainty (U_epis)', fontsize=11)
    ax2.set_xlabel('U_epis'); ax2.set_ylabel('Density')
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle(f'Uncertainty Distributions — {name}', fontsize=12)
    plt.tight_layout()
    path = f"{OUT}/plots/uncertainty_{fname(name)}.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

print("Plot functions ready")

Plot functions ready


In [4]:
def train_and_eval(model, train_loader, val_loader, test_loader,
                   epochs=EPOCHS, lr=LR, name='Model', save_path=None):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-6)
    best_auc, best_state = 0.0, None
    train_losses, val_aucs = [], []

    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            feats  = batch["features"][0].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            out    = model(feats)
            if isinstance(out, tuple): out = out[0]
            loss = F.cross_entropy(out, labels, label_smoothing=0.05)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        sched.step()
        train_losses.append(total_loss/len(train_loader))

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for batch in val_loader:
                feats = batch["features"][0].to(DEVICE)
                out   = model(feats)
                if isinstance(out, tuple): out = out[0]
                vp.append(torch.softmax(out, dim=1)[0,1].item())
                vt.append(batch["labels"].item())
        val_auc = roc_auc_score(vt, vp)
        val_aucs.append(val_auc)
        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.clone() for k,v in model.state_dict().items()}
        if ep % 10 == 0:
            print(f"[{name}] Ep {ep:02d}/{epochs} | "
                  f"Loss: {train_losses[-1]:.4f} | "
                  f"Val AUC: {val_auc:.4f} | Best: {best_auc:.4f}")

    plot_training_curves(train_losses, val_aucs, name)

    model.load_state_dict(best_state)
    model.eval()
    tp_list, tt = [], []
    with torch.no_grad():
        for batch in test_loader:
            feats = batch["features"][0].to(DEVICE)
            out   = model(feats)
            if isinstance(out, tuple): out = out[0]
            tp_list.append(torch.softmax(out, dim=1)[0,1].item())
            tt.append(batch["labels"].item())

    preds        = [1 if p>0.5 else 0 for p in tp_list]
    acc          = accuracy_score(tt, preds)
    auc          = roc_auc_score(tt, tp_list)
    tn,fp,fn,tp  = confusion_matrix(tt, preds).ravel()
    prec         = tp/(tp+fp) if (tp+fp)>0 else 0
    rec          = tp/(tp+fn) if (tp+fn)>0 else 0
    fpr_c,tpr_c,_= roc_curve(tt, tp_list)
    fp2,mc       = calibration_curve(tt, tp_list, n_bins=10)
    ece          = float(np.mean(np.abs(fp2-mc)))

    plot_confusion(int(tn),int(fp),int(fn),int(tp), name)

    print(f"\n{'='*50}")
    print(f"  {name} — TEST RESULTS")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  AUC       : {auc:.4f}")
    print(f"  ECE       : {ece:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  TP={tp} TN={tn} FP={fp} FN={fn}")
    print(f"{'='*50}\n")
    if save_path: torch.save(best_state, save_path)

    return {"Model":name, "Accuracy":round(acc*100,2),
            "AUC":round(auc,4), "ECE":round(ece,4),
            "Precision":round(prec,4), "Recall":round(rec,4),
            "TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn),
            "_fpr":fpr_c.tolist(),"_tpr":tpr_c.tolist(),
            "_fp2":fp2.tolist(),"_mc":mc.tolist(),
            "_losses":train_losses,"_aucs":val_aucs}

print("train_and_eval ready")

train_and_eval ready


In [5]:
class ABMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, att_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.25))
        self.att_V = nn.Linear(hidden, att_dim)
        self.att_U = nn.Linear(hidden, att_dim)
        self.att_w = nn.Linear(att_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 64), nn.GELU(),
            nn.Dropout(0.25), nn.Linear(64, 2))

    def forward(self, feats):
        h = self.encoder(feats)
        a = torch.tanh(self.att_V(h)) * torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a), dim=0)
        z = (a * h).sum(dim=0, keepdim=True)
        return self.classifier(z)

abmil_result = train_and_eval(
    ABMIL(), train_loader, val_loader, test_loader,
    name='ABMIL',
    save_path=f'{OUT}/checkpoints/abmil_best.pt')

[ABMIL] Ep 10/20 | Loss: 0.5296 | Val AUC: 0.7984 | Best: 0.8354
[ABMIL] Ep 20/20 | Loss: 0.5371 | Val AUC: 0.8004 | Best: 0.8354
  Saved: /kaggle/working/plots/training_ABMIL.png
  Saved: /kaggle/working/plots/confusion_ABMIL.png

  ABMIL — TEST RESULTS
  Accuracy  : 85.00%
  AUC       : 0.9233
  ECE       : 0.3437
  Precision : 1.0000
  Recall    : 0.6250
  TP=25 TN=60 FP=0 FN=15



In [6]:
class CLAM_SB(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, att_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(), nn.Dropout(0.25))
        self.att_V = nn.Linear(hidden, att_dim)
        self.att_U = nn.Linear(hidden, att_dim)
        self.att_w = nn.Linear(att_dim, 1)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, feats):
        h = self.encoder(feats)
        a = torch.tanh(self.att_V(h)) * torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a), dim=0)
        z = (a * h).sum(dim=0, keepdim=True)
        return self.classifier(z)

clam_result = train_and_eval(
    CLAM_SB(), train_loader, val_loader, test_loader,
    name='CLAM-SB',
    save_path=f'{OUT}/checkpoints/clam_best.pt')

[CLAM-SB] Ep 10/20 | Loss: 0.4872 | Val AUC: 0.8704 | Best: 0.8745
[CLAM-SB] Ep 20/20 | Loss: 0.4998 | Val AUC: 0.8765 | Best: 0.8786
  Saved: /kaggle/working/plots/training_CLAM-SB.png
  Saved: /kaggle/working/plots/confusion_CLAM-SB.png

  CLAM-SB — TEST RESULTS
  Accuracy  : 90.00%
  AUC       : 0.9167
  ECE       : 0.3269
  Precision : 0.9412
  Recall    : 0.8000
  TP=32 TN=58 FP=2 FN=8



In [7]:
class VAEABMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, latent=128, att_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.25))
        self.mu_head     = nn.Linear(hidden, latent)
        self.logvar_head = nn.Linear(hidden, latent)
        self.logvar_head.bias.data.fill_(-2.0)
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden), nn.GELU(), nn.Linear(hidden, feat_dim))
        self.att_V = nn.Linear(hidden, att_dim)
        self.att_U = nn.Linear(hidden, att_dim)
        self.att_w = nn.Linear(att_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(latent, 64), nn.GELU(),
            nn.Dropout(0.25), nn.Linear(64, 2))

    def reparameterize(self, mu, logvar):
        if self.training:
            return mu + torch.exp(0.5*logvar)*torch.randn_like(mu)
        return mu

    def forward(self, feats):
        h      = self.encoder(feats)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h).clamp(-8, 4)
        z      = self.reparameterize(mu, logvar)
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a), dim=0)
        z_bag  = (a*z).sum(dim=0, keepdim=True)
        logits = self.classifier(z_bag)
        kl     = -0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp())
        recon  = F.mse_loss(self.decoder(z), feats)
        return logits, kl, recon

def train_vaeabmil(model, train_loader, val_loader, test_loader,
                   epochs=EPOCHS, lr=LR, name='VAEABMIL', save_path=None):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-6)
    best_auc, best_state = 0.0, None
    train_losses, val_aucs = [], []

    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0
        beta = min(1.0, ep/20) * 0.005
        for batch in train_loader:
            feats  = batch["features"][0].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits, kl, recon = model(feats)
            loss = (F.cross_entropy(logits, labels, label_smoothing=0.05)
                    + beta*kl + 0.005*recon)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        sched.step()
        train_losses.append(total_loss/len(train_loader))

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for batch in val_loader:
                feats = batch["features"][0].to(DEVICE)
                logits,_,_ = model(feats)
                vp.append(torch.softmax(logits,dim=1)[0,1].item())
                vt.append(batch["labels"].item())
        val_auc = roc_auc_score(vt, vp)
        val_aucs.append(val_auc)
        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
        if ep % 10 == 0:
            print(f"[{name}] Ep {ep:02d}/{epochs} | "
                  f"Loss: {train_losses[-1]:.4f} | "
                  f"Val AUC: {val_auc:.4f} | Best: {best_auc:.4f}")

    plot_training_curves(train_losses, val_aucs, name)
    model.load_state_dict(best_state)
    model.eval()
    tp_list, tt = [], []
    with torch.no_grad():
        for batch in test_loader:
            feats = batch["features"][0].to(DEVICE)
            logits,_,_ = model(feats)
            tp_list.append(torch.softmax(logits,dim=1)[0,1].item())
            tt.append(batch["labels"].item())

    preds        = [1 if p>0.5 else 0 for p in tp_list]
    acc          = accuracy_score(tt, preds)
    auc          = roc_auc_score(tt, tp_list)
    tn,fp,fn,tp  = confusion_matrix(tt, preds).ravel()
    prec         = tp/(tp+fp) if (tp+fp)>0 else 0
    rec          = tp/(tp+fn) if (tp+fn)>0 else 0
    fpr_c,tpr_c,_= roc_curve(tt, tp_list)
    fp2,mc       = calibration_curve(tt, tp_list, n_bins=10)
    ece          = float(np.mean(np.abs(fp2-mc)))

    plot_confusion(int(tn),int(fp),int(fn),int(tp), name)

    print(f"\n{'='*50}")
    print(f"  {name} — TEST RESULTS")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  AUC       : {auc:.4f}")
    print(f"  ECE       : {ece:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  TP={tp} TN={tn} FP={fp} FN={fn}")
    print(f"{'='*50}\n")
    if save_path: torch.save(best_state, save_path)
    return {"Model":name,"Accuracy":round(acc*100,2),
            "AUC":round(auc,4),"ECE":round(ece,4),
            "Precision":round(prec,4),"Recall":round(rec,4),
            "TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn),
            "_fpr":fpr_c.tolist(),"_tpr":tpr_c.tolist(),
            "_fp2":fp2.tolist(),"_mc":mc.tolist(),
            "_losses":train_losses,"_aucs":val_aucs}

vaeabmil_result = train_vaeabmil(
    VAEABMIL(), train_loader, val_loader, test_loader,
    name='VAEABMIL',
    save_path=f'{OUT}/checkpoints/vaeabmil_best.pt')

[VAEABMIL] Ep 10/20 | Loss: 0.5094 | Val AUC: 0.7819 | Best: 0.8621
[VAEABMIL] Ep 20/20 | Loss: 0.4852 | Val AUC: 0.8189 | Best: 0.8621
  Saved: /kaggle/working/plots/training_VAEABMIL.png
  Saved: /kaggle/working/plots/confusion_VAEABMIL.png

  VAEABMIL — TEST RESULTS
  Accuracy  : 88.00%
  AUC       : 0.9183
  ECE       : 0.3576
  Precision : 0.9375
  Recall    : 0.7500
  TP=30 TN=58 FP=2 FN=10



In [8]:
class URATMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, latent=128,
                 att_dim=128, mc_samples=20, gate=True):
        super().__init__()
        self.mc_samples = mc_samples
        self.gate       = gate
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.25))
        self.mu_head     = nn.Linear(hidden, latent)
        self.logvar_head = nn.Linear(hidden, latent)
        self.logvar_head.bias.data.fill_(-2.0)  # Run A setting
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden), nn.GELU(), nn.Linear(hidden, feat_dim))
        self.prototypes  = nn.Parameter(torch.randn(2, latent))
        self.att_V       = nn.Linear(hidden, att_dim)
        self.att_U       = nn.Linear(hidden, att_dim)
        self.att_w       = nn.Linear(att_dim, 1)
        self.temperature = nn.Parameter(torch.ones(1))
        self.classifier  = nn.Sequential(
            nn.LayerNorm(latent), nn.Linear(latent, 128),
            nn.GELU(), nn.Dropout(0.25), nn.Linear(128, 2))

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5*logvar)*torch.randn_like(mu)

    def forward(self, feats):
        h      = self.encoder(feats)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h).clamp(-8, 4)

        if self.training:
            z      = self.reparameterize(mu, logvar)
            u_alea = torch.exp(logvar).mean(dim=1, keepdim=True)
            u_epis = torch.zeros_like(u_alea)
        else:
            samples = torch.stack([
                self.reparameterize(mu, logvar)
                for _ in range(self.mc_samples)], dim=0)
            z      = samples.mean(dim=0)
            u_alea = torch.exp(logvar).mean(dim=1, keepdim=True)
            u_epis = samples.var(dim=0).mean(dim=1, keepdim=True)

        u_total = u_alea + u_epis
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a), dim=0)

        if self.gate:
            a = a / (1.0 + u_total)
            a = a / (a.sum() + 1e-8)

        z_bag  = (a*z).sum(dim=0, keepdim=True)
        logits = self.classifier(z_bag)/self.temperature.clamp(min=0.1)
        kl     = -0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp())
        recon  = F.mse_loss(self.decoder(z), feats)
        dists  = torch.cdist(z, self.prototypes)
        ood_s  = dists.min(dim=1).values.mean()
        return logits, kl, recon, ood_s, u_alea.mean(), u_epis.mean()


def train_urat(model, train_loader, val_loader, test_loader,
               epochs=EPOCHS, lr=LR, name='URAT-MIL',
               lam_kl=0.005, lam_recon=0.005, lam_ood=0.005,
               save_path=None):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-6)
    best_auc, best_state = 0.0, None
    train_losses, val_aucs = [], []

    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0
        beta = min(1.0, ep/15)*lam_kl
        for batch in train_loader:
            feats  = batch["features"][0].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits,kl,recon,ood_s,_,_ = model(feats)
            loss = (F.cross_entropy(logits, labels, label_smoothing=0.05)
                    + beta*kl + lam_recon*recon + lam_ood*ood_s)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        sched.step()
        train_losses.append(total_loss/len(train_loader))

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for batch in val_loader:
                feats = batch["features"][0].to(DEVICE)
                logits,*_ = model(feats)
                vp.append(torch.softmax(logits,dim=1)[0,1].item())
                vt.append(batch["labels"].item())
        val_auc = roc_auc_score(vt, vp)
        val_aucs.append(val_auc)
        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
        if ep % 10 == 0:
            print(f"[{name}] Ep {ep:02d}/{epochs} | "
                  f"Loss: {train_losses[-1]:.4f} | "
                  f"Val AUC: {val_auc:.4f} | Best: {best_auc:.4f}")

    plot_training_curves(train_losses, val_aucs, name)
    model.load_state_dict(best_state)
    model.eval()
    tp_list, tt, ualeas, uepiss = [], [], [], []
    with torch.no_grad():
        for batch in test_loader:
            feats = batch["features"][0].to(DEVICE)
            logits,_,_,_,ua,ue = model(feats)
            tp_list.append(torch.softmax(logits,dim=1)[0,1].item())
            tt.append(batch["labels"].item())
            ualeas.append(ua.item())
            uepiss.append(ue.item())

    preds        = [1 if p>0.5 else 0 for p in tp_list]
    acc          = accuracy_score(tt, preds)
    auc          = roc_auc_score(tt, tp_list)
    tn,fp,fn,tp  = confusion_matrix(tt, preds).ravel()
    prec         = tp/(tp+fp) if (tp+fp)>0 else 0
    rec          = tp/(tp+fn) if (tp+fn)>0 else 0
    fpr_c,tpr_c,_= roc_curve(tt, tp_list)
    fp2,mc       = calibration_curve(tt, tp_list, n_bins=10)
    ece          = float(np.mean(np.abs(fp2-mc)))
    mean_ua      = float(np.mean(ualeas))
    mean_ue      = float(np.mean(uepiss))

    plot_confusion(int(tn),int(fp),int(fn),int(tp), name)
    plot_uncertainty(ualeas, uepiss, tt, name)

    print(f"\n{'='*50}")
    print(f"  {name} — TEST RESULTS")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  AUC       : {auc:.4f}")
    print(f"  ECE       : {ece:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  U_alea    : {mean_ua:.4f}")
    print(f"  U_epis    : {mean_ue:.4f}")
    print(f"  TP={tp} TN={tn} FP={fp} FN={fn}")
    print(f"{'='*50}\n")
    if save_path: torch.save(best_state, save_path)
    return {"Model":name,"Accuracy":round(acc*100,2),
            "AUC":round(auc,4),"ECE":round(ece,4),
            "Precision":round(prec,4),"Recall":round(rec,4),
            "U_alea":round(mean_ua,4),"U_epis":round(mean_ue,4),
            "TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn),
            "_fpr":fpr_c.tolist(),"_tpr":tpr_c.tolist(),
            "_fp2":fp2.tolist(),"_mc":mc.tolist(),
            "_losses":train_losses,"_aucs":val_aucs,
            "_u_alea":ualeas,"_u_epis":uepiss,"_labels":tt}

urat_result = train_urat(
    URATMIL(gate=True),
    train_loader, val_loader, test_loader,
    name='URAT-MIL (Proposed)',
    lam_kl=0.005, lam_recon=0.005, lam_ood=0.005,
    save_path=f'{OUT}/checkpoints/urat_full_best.pt')

[URAT-MIL (Proposed)] Ep 10/20 | Loss: 0.6003 | Val AUC: 0.8477 | Best: 0.8663
[URAT-MIL (Proposed)] Ep 20/20 | Loss: 0.4738 | Val AUC: 0.8519 | Best: 0.8663
  Saved: /kaggle/working/plots/training_URAT-MIL_Proposed.png
  Saved: /kaggle/working/plots/confusion_URAT-MIL_Proposed.png
  Saved: /kaggle/working/plots/uncertainty_URAT-MIL_Proposed.png

  URAT-MIL (Proposed) — TEST RESULTS
  Accuracy  : 88.00%
  AUC       : 0.9225
  ECE       : 0.2670
  Precision : 0.9667
  Recall    : 0.7250
  U_alea    : 0.0146
  U_epis    : 0.0146
  TP=29 TN=59 FP=1 FN=11



In [9]:
nogate_result = train_urat(
    URATMIL(gate=False),
    train_loader, val_loader, test_loader,
    name='eval_nogate (No Gating)',
    lam_kl=0.005, lam_recon=0.005, lam_ood=0.005,
    save_path=f'{OUT}/checkpoints/urat_nogate_best.pt')

[eval_nogate (No Gating)] Ep 10/20 | Loss: 0.5632 | Val AUC: 0.8642 | Best: 0.8663
[eval_nogate (No Gating)] Ep 20/20 | Loss: 0.5199 | Val AUC: 0.8519 | Best: 0.8663
  Saved: /kaggle/working/plots/training_eval_nogate_No_Gating.png
  Saved: /kaggle/working/plots/confusion_eval_nogate_No_Gating.png
  Saved: /kaggle/working/plots/uncertainty_eval_nogate_No_Gating.png

  eval_nogate (No Gating) — TEST RESULTS
  Accuracy  : 88.00%
  AUC       : 0.9217
  ECE       : 0.2866
  Precision : 0.9375
  Recall    : 0.7500
  U_alea    : 0.0160
  U_epis    : 0.0160
  TP=30 TN=58 FP=2 FN=10



In [10]:
all_results = [abmil_result, clam_result, vaeabmil_result,
               nogate_result, urat_result]

# Combined ROC
roc_dict = {r['Model']:(r['_fpr'],r['_tpr'],r['AUC']) for r in all_results}
plot_all_roc(roc_dict)

# Reliability diagrams
rel_dict = {r['Model']:(r['_fp2'],r['_mc'],r['ECE']) for r in all_results}
plot_reliability(rel_dict)

# Ablation bar chart
plot_ablation_bar(all_results)

# Clean results table
clean_keys = ["Model","Accuracy","AUC","ECE","Precision","Recall",
              "U_alea","U_epis","TP","TN","FP","FN"]
clean = [{k: r.get(k,"N/A") for k in clean_keys} for r in all_results]
df = pd.DataFrame(clean)

print("\n" + "="*70)
print("FINAL RESULTS TABLE")
print("="*70)
print(df[["Model","Accuracy","AUC","ECE","Precision","Recall",
          "U_alea","U_epis"]].to_string(index=False))
print("="*70)

df.to_csv(f"{OUT}/metrics/final_results.csv", index=False)

# Full JSON
with open(f"{OUT}/metrics/full_results.json",'w') as f:
    json.dump(all_results, f, indent=2)

# Experiment metadata
meta = {
    "timestamp":   datetime.datetime.now().isoformat(),
    "dataset":     "CAMELYON16",
    "source":      "DSMIL Camelyon16 Feature (ForcewithMe, Kaggle)",
    "n_train":     len(train_rec),
    "n_val":       len(val_rec),
    "n_test":      len(test_rec),
    "test_pos":    sum(r[1] for r in test_rec),
    "max_patches": MAX_PATCHES,
    "epochs":      EPOCHS,
    "lr":          LR,
    "seed":        SEED,
    "lam_kl":      0.005,
    "lam_recon":   0.005,
    "lam_ood":     0.005,
    "device":      DEVICE,
}
with open(f"{OUT}/metrics/experiment_metadata.json",'w') as f:
    json.dump(meta, f, indent=2)

# List all outputs
print("\nALL OUTPUT FILES ")
for folder in ['checkpoints','plots','metrics']:
    path = f"{OUT}/{folder}"
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"\n{folder}/")
        for fname_f in sorted(files):
            size = os.path.getsize(f"{path}/{fname_f}")
            print(f"  {fname_f}  ({size/1024:.1f} KB)")


  Saved: /kaggle/working/plots/roc_all_models.png
  Saved: /kaggle/working/plots/reliability_all.png
  Saved: /kaggle/working/plots/ablation_comparison.png

FINAL RESULTS TABLE
                  Model  Accuracy    AUC    ECE  Precision  Recall  U_alea  U_epis
                  ABMIL      85.0 0.9233 0.3437     1.0000   0.625     N/A     N/A
                CLAM-SB      90.0 0.9167 0.3269     0.9412   0.800     N/A     N/A
               VAEABMIL      88.0 0.9183 0.3576     0.9375   0.750     N/A     N/A
eval_nogate (No Gating)      88.0 0.9217 0.2866     0.9375   0.750   0.016   0.016
    URAT-MIL (Proposed)      88.0 0.9225 0.2670     0.9667   0.725  0.0146  0.0146

ALL OUTPUT FILES 

checkpoints/
  abmil_best.pt  (842.4 KB)
  clam_best.pt  (776.5 KB)
  urat_full_best.pt  (1748.5 KB)
  urat_nogate_best.pt  (1748.6 KB)
  vaeabmil_best.pt  (1712.6 KB)

plots/
  ablation_comparison.png  (266.3 KB)
  confusion_ABMIL.png  (67.0 KB)
  confusion_CLAM-SB.png  (66.1 KB)
  confusion_URAT-MIL_Pr